# Session 7 · Exploring Hadoop Architecture and Components

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-07-hadoop-architecture-and-components.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 1 · Replication vs erasure coding

In [1]:
schemes = {
    "replication x3": dict(data=1, parity=2, tolerates=2),
    "replication x2": dict(data=1, parity=1, tolerates=1),
    "EC RS-3-2":      dict(data=3, parity=2, tolerates=2),
    "EC RS-6-3":      dict(data=6, parity=3, tolerates=3),
    "EC RS-10-4":     dict(data=10, parity=4, tolerates=4),
}
logical_tb = 500
print(f"{'scheme':<16}{'overhead':>9}{'raw TB for 500 TB':>19}{'failures survived':>19}")
for name, s in schemes.items():
    overhead = s["parity"] / s["data"]
    print(f"{name:<16}{overhead:>8.0%}{logical_tb * (1 + overhead):>19.0f}{s['tolerates']:>19}")

scheme           overhead  raw TB for 500 TB  failures survived
replication x3      200%               1500                  2
replication x2      100%               1000                  1
EC RS-3-2            67%                833                  2
EC RS-6-3            50%                750                  3
EC RS-10-4           40%                700                  4


### Example 2 · Parity in action (the idea behind erasure coding)

In [2]:
cells = [b"HADO", b"OP-3", b"-EC!"]                       # 3 data cells on 3 DataNodes
parity = bytes(a ^ b ^ c for a, b, c in zip(*cells))      # 1 parity cell on a 4th DataNode
print("data cells :", cells)
print("parity cell:", parity.hex())

lost = 1
print(f"\nDataNode holding cell {lost} fails!")
survivors = [c for i, c in enumerate(cells) if i != lost] + [parity]
rebuilt = bytes(x ^ y ^ z for x, y, z in zip(*survivors))
print("rebuilt cell:", rebuilt, "| correct:", rebuilt == cells[lost])
print(f"storage overhead: {1/len(cells):.0%} (vs 200% for 3 replicas)")

data cells : [b'HADO', b'OP-3', b'-EC!']
parity cell: 2a542a5d

DataNode holding cell 1 fails!
rebuilt cell: b'OP-3' | correct: True
storage overhead: 33% (vs 200% for 3 replicas)


### Example 3 · NameNode failover with a journal quorum

In [3]:
journal = {"jn1": [], "jn2": [], "jn3": []}
state = {"nn1": "active", "nn2": "standby"}
standby_ns = []

def write_edit(op):
    acks = 0
    for jn, log in journal.items():
        if jn != "jn3" or len(log) < 2:        # jn3 goes offline after 2 edits
            log.append(op); acks += 1
    ok = acks >= 2                              # majority of 3
    print(f"edit {op:<22} acks={acks}/3 -> {'committed' if ok else 'FAILED'}")

def standby_tail():
    longest = max(journal.values(), key=len)
    standby_ns.extend(longest[len(standby_ns):])

for op in ["mkdir /data", "create /data/a.csv", "create /data/b.csv", "delete /data/a.csv"]:
    write_edit(op)
    standby_tail()

print("\n*** ZKFC on nn1: health check failed, ZooKeeper lock released ***")
state["nn1"] = "fenced"
state["nn2"] = "active"
print("NameNode states:", state)
print("nn2 namespace edits:", standby_ns)

edit mkdir /data            acks=3/3 -> committed
edit create /data/a.csv     acks=3/3 -> committed
edit create /data/b.csv     acks=2/3 -> committed
edit delete /data/a.csv     acks=2/3 -> committed

*** ZKFC on nn1: health check failed, ZooKeeper lock released ***
NameNode states: {'nn1': 'fenced', 'nn2': 'active'}
nn2 namespace edits: ['mkdir /data', 'create /data/a.csv', 'create /data/b.csv', 'delete /data/a.csv']


### Example 4 · HBase-style row keys

In [4]:
MAX_TS = 9_999_999_999
readings = [("meter042", 1729760000, 3.1), ("meter007", 1729760300, 1.2),
            ("meter042", 1729760600, 3.4), ("meter042", 1729760900, 2.9),
            ("meter007", 1729761200, 1.5)]

table = {f"{dev}#{MAX_TS - ts}": {"cf:kwh": kwh, "cf:ts": ts} for dev, ts, kwh in readings}
print("rows as HBase stores them (sorted by row key):")
for key in sorted(table):
    print(f"  {key}  {table[key]}")

prefix = "meter042#"                                # a prefix scan = all readings for one meter
latest = [table[k] for k in sorted(table) if k.startswith(prefix)][:2]
print("\nlatest 2 readings for meter042:", [(r['cf:ts'], r['cf:kwh']) for r in latest])

rows as HBase stores them (sorted by row key):
  meter007#8270238799  {'cf:kwh': 1.5, 'cf:ts': 1729761200}
  meter007#8270239699  {'cf:kwh': 1.2, 'cf:ts': 1729760300}
  meter042#8270239099  {'cf:kwh': 2.9, 'cf:ts': 1729760900}
  meter042#8270239399  {'cf:kwh': 3.4, 'cf:ts': 1729760600}
  meter042#8270239999  {'cf:kwh': 3.1, 'cf:ts': 1729760000}

latest 2 readings for meter042: [(1729760900, 2.9), (1729760600, 3.4)]
